# Proyecto de Regresión Lineal Múltiple
## Predicción de Ingresos Recaudados por Caja — Gobierno Regional del Callao (Enero–Marzo 2026)

**Curso:** Minería de Datos  
**Docente:** Dr. José Herrera  
**Universidad:** UNMSM — Facultad de Ingeniería de Sistemas e Informática  
**Fuente del dataset:** Datos Abiertos del Estado Peruano — datosabiertos.gob.pe / GORECALLAO  
**URL:** https://www.datosabiertos.gob.pe/dataset/ingresos-recaudados-por-caja-del-gobierno-regional-del-callao

---
## 1. Planteamiento del problema

El Gobierno Regional del Callao (GORECALLAO) registra diariamente los ingresos recaudados en caja por distintos conceptos: licencias de conducir, multas, matrículas, ventas de terrenos, entre otros. Una gestión eficiente requiere anticipar cuánto se recaudará en los próximos días para planificar recursos y tesorería.

**Pregunta de minería:** ¿Es posible predecir el ingreso diario total recaudado por caja del GORECALLAO en función del día cronológico, el mes y el día de la semana?

**Variable dependiente (Y):** Total de ingresos recaudados en el día (en soles, S/)  
**Variables independientes (X):** Número de día cronológico, mes y día de semana (codificado con dummies)  
**Tipo de modelo:** Regresión lineal múltiple (OLS)

---
## 2. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import linregress
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Librerías cargadas correctamente.')

---
## 3. Carga y descripción del dataset

In [ ]:
df = pd.read_csv('Ingresos_recaudados_por_caja_enero_a_marzo_2026.csv',
                 encoding='utf-8', sep=None, engine='python')
df.columns = [c.lstrip('\ufeff') for c in df.columns]
df['FECHA'] = pd.to_datetime(df['FECHA'].astype(str), format='%Y%m%d')

print(f'Dimensiones del dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Periodo: {df["FECHA"].min().date()} a {df["FECHA"].max().date()}')
df.head(5)

In [ ]:
# Diccionario de variables
diccionario = pd.DataFrame({
    'Variable': ['FECHA_CORTE','AÑO','MES','FECHA','TIPO_DOC','DOCUMENTO','GLOSA','CONCEPTO','TOTAL'],
    'Descripción': [
        'Día en que se generó el dataset (aaaammdd)',
        'Año de la recaudación',
        'Mes de la recaudación (1-12)',
        'Fecha de la recaudación (aaaammdd)',
        'Tipo de documento emitido (Boleta, Factura, Recibo, Ticket)',
        'Número de recibo de ingreso',
        'Detalle descriptivo de lo recaudado',
        'Concepto de recaudación',
        'Importe recaudado (S/)'
    ],
    'Tipo': ['Numérico','Numérico','Numérico','Fecha','Texto','Texto','Texto','Texto','Numérico']
})
diccionario

---
## 4. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Estadísticos descriptivos del TOTAL
print('=== Estadísticos por transacción individual ===')
print(df['TOTAL'].describe().round(2))
print(f'\nNúmero de conceptos únicos: {df["CONCEPTO"].nunique()}')
print(f'Número de tipos de documento: {df["TIPO_DOC"].nunique()}')

In [ ]:
# Recaudación por mes
por_mes = df.groupby('MES')['TOTAL'].agg(['sum','count','mean']).round(2)
por_mes.index = ['Enero','Febrero','Marzo']
por_mes.columns = ['Total (S/)','N° transacciones','Promedio por transacción (S/)']
print('=== Recaudación por mes ===')
print(por_mes.to_string())

In [ ]:
# Top 10 conceptos por recaudación total
top10 = df.groupby('CONCEPTO')['TOTAL'].sum().sort_values(ascending=False).head(10)
print('=== Top 10 conceptos por recaudación total ===')
for concepto, monto in top10.items():
    print(f'  S/ {monto:>12,.2f}  |  {concepto[:70]}')

In [ ]:
# Agregación diaria
daily = df.groupby('FECHA')['TOTAL'].sum().reset_index()
daily['DIA_NUM']    = range(1, len(daily)+1)
daily['DIA_SEMANA'] = daily['FECHA'].dt.dayofweek   # 0=Lun, 6=Dom
daily['MES']        = daily['FECHA'].dt.month
daily['ES_LABORAL'] = (daily['DIA_SEMANA'] < 5).astype(int)

print(f'Días únicos en el dataset: {len(daily)}')
print(f'Días laborales: {daily["ES_LABORAL"].sum()}')
print(f'Fines de semana: {(~daily["ES_LABORAL"].astype(bool)).sum()}')
print(f'\nTotal recaudado en el periodo: S/ {daily["TOTAL"].sum():,.2f}')
print(f'Promedio diario: S/ {daily["TOTAL"].mean():,.2f}')
print(f'Máximo diario:   S/ {daily["TOTAL"].max():,.2f} ({daily.loc[daily["TOTAL"].idxmax(), "FECHA"].date()})')
print(f'Mínimo diario:   S/ {daily["TOTAL"].min():,.2f} ({daily.loc[daily["TOTAL"].idxmin(), "FECHA"].date()})')

In [ ]:
# --- EDA visual ---
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('EDA — Ingresos diarios recaudados por caja GORECALLAO (Ene–Mar 2026)',
             fontsize=13, fontweight='bold')

# 1. Serie temporal diaria
axes[0,0].plot(daily['FECHA'], daily['TOTAL'], color='steelblue', linewidth=1.2)
axes[0,0].fill_between(daily['FECHA'], daily['TOTAL'], alpha=0.15, color='steelblue')
axes[0,0].set_title('Serie temporal de ingresos diarios')
axes[0,0].set_xlabel('Fecha')
axes[0,0].set_ylabel('Total (S/)')
axes[0,0].tick_params(axis='x', rotation=30)

# 2. Promedio por día de semana
nombres_dia = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']
prom_semana = daily.groupby('DIA_SEMANA')['TOTAL'].mean()
colores = ['steelblue']*5 + ['coral','salmon']
axes[0,1].bar(nombres_dia, prom_semana.values, color=colores)
axes[0,1].set_title('Promedio de ingresos por día de semana')
axes[0,1].set_ylabel('Promedio (S/)')

# 3. Distribución del total diario
axes[1,0].hist(daily['TOTAL'], bins=20, color='steelblue', edgecolor='white', alpha=0.8)
axes[1,0].axvline(daily['TOTAL'].mean(), color='red', linestyle='--', label=f'Media: S/{daily["TOTAL"].mean():,.0f}')
axes[1,0].set_title('Distribución del ingreso diario total')
axes[1,0].set_xlabel('Total diario (S/)')
axes[1,0].set_ylabel('Frecuencia')
axes[1,0].legend()

# 4. Top 5 conceptos
top5 = df.groupby('CONCEPTO')['TOTAL'].sum().sort_values(ascending=True).tail(5)
etiquetas = [c[:35]+'...' if len(c)>35 else c for c in top5.index]
axes[1,1].barh(etiquetas, top5.values, color='teal')
axes[1,1].set_title('Top 5 conceptos por recaudación total')
axes[1,1].set_xlabel('Total (S/)')

plt.tight_layout()
plt.savefig('eda_ingresos.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlación entre variables numéricas
corr_df = daily[['DIA_NUM','MES','DIA_SEMANA','TOTAL']].copy()
print('=== Matriz de correlación ===')
print(corr_df.corr().round(4))

fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(corr_df.corr(), annot=True, fmt='.3f', cmap='coolwarm',
            center=0, ax=ax, square=True)
ax.set_title('Mapa de correlación entre variables')
plt.tight_layout()
plt.savefig('correlacion.png', bbox_inches='tight')
plt.show()

---
## 5. Preparación de variables para la regresión

In [ ]:
# Codificación dummies para día de semana (lunes=referencia)
dummies = pd.get_dummies(daily['DIA_SEMANA'], prefix='ds', drop_first=True)
X = pd.concat([daily[['DIA_NUM', 'MES']], dummies], axis=1)
y = daily['TOTAL']

print('Variables del modelo:')
print(f'  Predictores (X): {list(X.columns)}')
print(f'  Variable objetivo (y): TOTAL')
print(f'  Observaciones: {len(X)}')
X.head()

---
## 6. Modelado: Regresión Lineal Múltiple

In [ ]:
# División 80/20 (estratificada por mes para representatividad)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Tamaño entrenamiento: {len(X_train)} días ({len(X_train)/len(X)*100:.0f}%)')
print(f'Tamaño prueba:        {len(X_test)} días ({len(X_test)/len(X)*100:.0f}%)')

In [ ]:
# Entrenamiento
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Coeficientes
print('=== Coeficientes del modelo ===')
print(f'  Intercepto (a):  S/ {modelo.intercept_:,.2f}')
nombres_var = ['Día num. (tendencia)','Mes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']
for nombre, coef in zip(nombres_var, modelo.coef_):
    print(f'  {nombre:<28}: {coef:>+10,.2f}')

print()
print('Interpretación de la ecuación:')
print('  Y = 54,378 + 332·(DIA_NUM) - 13,365·(MES) + [ajuste por día de semana]')

---
## 7. Evaluación del modelo

In [ ]:
# Métricas en entrenamiento y prueba
y_pred_train = modelo.predict(X_train)
y_pred_test  = modelo.predict(X_test)

r2_train = modelo.score(X_train, y_train)
r2_test  = modelo.score(X_test, y_test)

mae_test  = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mape_test = np.mean(np.abs((y_test - y_pred_test) / y_test)) * 100

print('=== Métricas de evaluación ===')
print(f'  R² entrenamiento:  {r2_train:.4f}')
print(f'  R² prueba:         {r2_test:.4f}')
print(f'  MAE  (prueba):     S/ {mae_test:,.2f}')
print(f'  RMSE (prueba):     S/ {rmse_test:,.2f}')
print(f'  MAPE (prueba):     {mape_test:.1f}%')

In [ ]:
# Validación cruzada k=5
modelo_cv = LinearRegression()
r2_scores = cross_val_score(modelo_cv, X, y, cv=5, scoring='r2')
rmse_scores = -cross_val_score(modelo_cv, X, y, cv=5, scoring='neg_root_mean_squared_error')

print('=== Validación cruzada (k=5) ===')
print(f'  R² por pliegue:   {[round(s,4) for s in r2_scores]}')
print(f'  R² promedio:      {r2_scores.mean():.4f} ± {r2_scores.std():.4f}')
print(f'  RMSE promedio:    S/ {rmse_scores.mean():,.2f}')
print()
if r2_test - r2_train < 0.05:
    print('  → No hay indicios de sobreajuste (diferencia R² train/test < 0.05)')
else:
    print('  → Posible sobreajuste (diferencia R² train/test ≥ 0.05)')

In [ ]:
# Gráficos de diagnóstico
modelo_final = LinearRegression().fit(X, y)
y_pred_all = modelo_final.predict(X)
residuos = y - y_pred_all

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Diagnóstico del modelo de regresión lineal múltiple', fontsize=12, fontweight='bold')

# Real vs Predicho
axes[0].scatter(y, y_pred_all, alpha=0.6, color='steelblue', edgecolors='white', linewidth=0.5)
min_val, max_val = min(y.min(), y_pred_all.min()), max(y.max(), y_pred_all.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Y = Ŷ')
axes[0].set_xlabel('Valores reales (S/)')
axes[0].set_ylabel('Valores predichos (S/)')
axes[0].set_title('Real vs. Predicho')
axes[0].legend()

# Residuos vs. DIA_NUM
axes[1].scatter(daily['DIA_NUM'], residuos, alpha=0.6, color='teal', edgecolors='white', linewidth=0.5)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Día cronológico')
axes[1].set_ylabel('Residuo (S/)')
axes[1].set_title('Residuos a lo largo del tiempo')

# Distribución de residuos
axes[2].hist(residuos, bins=15, color='steelblue', edgecolor='white', alpha=0.8)
axes[2].axvline(0, color='red', linestyle='--')
axes[2].set_xlabel('Residuo (S/)')
axes[2].set_ylabel('Frecuencia')
axes[2].set_title('Distribución de residuos')

plt.tight_layout()
plt.savefig('diagnostico_modelo.png', bbox_inches='tight')
plt.show()

---
## 8. Predicciones para abril 2026

In [ ]:
# Primera semana de abril 2026 (días laborales: lun 6, mar 7, mié 8, jue 9, vie 10 abril)
# DIA_NUM continúa desde 90 (el último día fue 89 = 31 mar)
# ds_0=Lun (referencia), ds_1=Mar, ds_2=Mié, ds_3=Jue, ds_4=Vie, ds_5=Sáb, ds_6=Dom

proximos = [
    {'fecha': '2026-04-01', 'dia_sem': 'Miércoles', 'DIA_NUM': 90, 'MES': 4, 'ds_1':0,'ds_2':1,'ds_3':0,'ds_4':0,'ds_5':0,'ds_6':0},
    {'fecha': '2026-04-02', 'dia_sem': 'Jueves',    'DIA_NUM': 91, 'MES': 4, 'ds_1':0,'ds_2':0,'ds_3':1,'ds_4':0,'ds_5':0,'ds_6':0},
    {'fecha': '2026-04-03', 'dia_sem': 'Viernes',   'DIA_NUM': 92, 'MES': 4, 'ds_1':0,'ds_2':0,'ds_3':0,'ds_4':1,'ds_5':0,'ds_6':0},
    {'fecha': '2026-04-06', 'dia_sem': 'Lunes',     'DIA_NUM': 93, 'MES': 4, 'ds_1':0,'ds_2':0,'ds_3':0,'ds_4':0,'ds_5':0,'ds_6':0},
    {'fecha': '2026-04-07', 'dia_sem': 'Martes',    'DIA_NUM': 94, 'MES': 4, 'ds_1':1,'ds_2':0,'ds_3':0,'ds_4':0,'ds_5':0,'ds_6':0},
]

pred_df = pd.DataFrame(proximos)
X_pred = pred_df[['DIA_NUM','MES','ds_1','ds_2','ds_3','ds_4','ds_5','ds_6']]
pred_df['Predicción (S/)'] = modelo_final.predict(X_pred).round(2)

print('=== Predicciones — Primera semana laboral de abril 2026 ===')
print(pred_df[['fecha','dia_sem','Predicción (S/)']].to_string(index=False))
print(f'\nTotal estimado semana: S/ {pred_df["Predicción (S/)"].sum():,.2f}')

In [ ]:
# Gráfico de predicciones
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(daily['FECHA'], daily['TOTAL'], color='steelblue', linewidth=1.2, label='Datos reales', alpha=0.8)
ax.plot(daily['FECHA'], y_pred_all, color='orange', linewidth=1.5, linestyle='--', label='Modelo (ajuste)', alpha=0.9)

fechas_pred = pd.to_datetime([d['fecha'] for d in proximos])
preds_vals  = pred_df['Predicción (S/)'].values
ax.plot(fechas_pred, preds_vals, 'o-', color='red', linewidth=2, markersize=8, label='Predicción abril 2026')

ax.axvline(pd.Timestamp('2026-03-31'), color='gray', linestyle=':', linewidth=1)
ax.text(pd.Timestamp('2026-03-31'), ax.get_ylim()[1]*0.95, '  Fin datos', fontsize=9, color='gray')

ax.set_title('Ingresos diarios reales, ajuste del modelo y predicciones abril 2026', fontsize=12)
ax.set_xlabel('Fecha')
ax.set_ylabel('Total diario (S/)')
ax.legend()
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('predicciones.png', bbox_inches='tight')
plt.show()

---
## 9. Conclusiones y limitaciones

### Hallazgos principales
- El modelo de regresión lineal múltiple explica aproximadamente el **27.9% (R² = 0.279)** de la variabilidad del ingreso diario, incorporando el día cronológico, el mes y el día de la semana.
- El **día de la semana es el predictor más relevante**: los miércoles y jueves concentran mayor recaudación (S/ ~40 000–56 000), mientras que sábados y domingos caen drásticamente (S/ ~3 000–10 000).
- Se observa una **leve tendencia positiva** (+S/ 332 por día transcurrido), aunque con alta variabilidad.
- Los **tres principales conceptos** de recaudación son: Examen de manejo (S/ 825,961), Normas de transporte (S/ 647,438) y Licencia de conducir (S/ 387,367).

### Limitaciones
1. **R² moderado (0.28):** La mayor parte de la variabilidad se debe a factores no incluidos en el modelo: eventos especiales, vencimiento de plazos de pago, campañas institucionales.
2. **Dataset de corto plazo:** Solo se cuenta con 89 días (un trimestre). Un modelo robusto requeriría al menos 2–3 años de datos para capturar estacionalidades anuales.
3. **Valores atípicos:** Algunos días presentan recaudaciones muy superiores al promedio (hasta S/ 260,000) vinculadas a pagos de multas laborales o eventos puntuales, lo que infla el RMSE.
4. **Supuesto de linealidad:** La relación entre el tiempo y los ingresos puede no ser estrictamente lineal; modelos de series de tiempo (ARIMA, Prophet) podrían ser más apropiados.

### Recomendación
Para mejorar el modelo se recomienda incorporar variables adicionales como: número de días hábiles restantes en el mes, si el día es fin de plazo para algún trámite, y el volumen de atenciones previstas en las oficinas de GORECALLAO.

In [ ]:
# Resumen ejecutivo del modelo
resumen = {
    'Métrica': ['R² (entrenamiento)', 'R² (prueba)', 'R² CV promedio (k=5)', 'MAE', 'RMSE', 'MAPE'],
    'Valor': [f'{r2_train:.4f}', f'{r2_test:.4f}', f'{r2_scores.mean():.4f} ± {r2_scores.std():.4f}',
              f'S/ {mae_test:,.2f}', f'S/ {rmse_test:,.2f}', f'{mape_test:.1f}%']
}
pd.DataFrame(resumen)